### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import warnings

from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.nn_regressor import NNRegressor
from src.cnn_regressor import CNNRegressor
from src.gb_classifier import GBClassifier

warnings.filterwarnings("ignore")

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = modeling_config["main"]["active_dataset"]
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, test = data_manager.load_image_data(active_dataset)

X, y = processor.split_features_target(train)
X_test, y_test = processor.split_features_target(test)

y, y_test = y.reshape(-1, 1), y_test.reshape(-1, 1)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

### EVALUATION FUNCTION

In [3]:
def fit_and_evaluate(model, flatten=False, **extra_params):
    xt, xv, xte = X_train, X_valid, X_test

    if flatten:
        xt = xt.reshape(xt.shape[0], -1)
        xv = xv.reshape(xv.shape[0], -1)
        xte = xte.reshape(xte.shape[0], -1)

    model.fit(xt, y_train, **extra_params)

    valid_prob_preds = model.predict_proba(xv)
    valid_class_preds = np.argmax(valid_prob_preds, axis=1)

    valid_log_loss = log_loss(y_valid, valid_prob_preds, labels=np.unique(y_train))
    valid_accuracy = accuracy_score(y_valid, valid_class_preds)

    test_prob_preds = model.predict_proba(xte)
    test_class_preds = np.argmax(test_prob_preds, axis=1)

    test_log_loss = log_loss(y_test, test_prob_preds, labels=np.unique(y_train))
    test_accuracy = accuracy_score(y_test, test_class_preds)
    
    return valid_log_loss, valid_accuracy, test_log_loss, test_accuracy

### LOGISTIC REGRESSION

In [8]:
%%time

logistic = LogisticRegression(C=1.0, random_state=42)
logistic_valid_log_loss, logistic_valid_accuracy, logistic_test_log_loss, logistic_test_accuracy = fit_and_evaluate(logistic, flatten=True)

CPU times: total: 3min 48s
Wall time: 25.8 s


In [9]:
print(f"Logistic Valid Log Loss: {logistic_valid_log_loss:.6f}")
print(f"Logistic Valid Accuracy: {logistic_valid_accuracy:.6f}")
print("-" * 50)
print(f"Logistic Test Log Loss:  {logistic_test_log_loss:.6f}")
print(f"Logistic Test Accuracy:  {logistic_test_accuracy:.6f}")

Logistic Valid Log Loss: 1.727530
Logistic Valid Accuracy: 0.406200
--------------------------------------------------
Logistic Test Log Loss:  1.728494
Logistic Test Accuracy:  0.403000


### DECISION TREE

In [10]:
%%time

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_valid_log_loss, dt_valid_accuracy, dt_test_log_loss, dt_test_accuracy = fit_and_evaluate(dt, flatten=True)

CPU times: total: 38.8 s
Wall time: 38.9 s


In [11]:
print(f"Decision Tree Valid Log Loss: {dt_valid_log_loss:.4f}")
print(f"Decision Tree Valid Accuracy: {dt_valid_accuracy:.4f}")
print("-" * 50)
print(f"Decision Tree Test Log Loss:  {dt_test_log_loss:.4f}")
print(f"Decision Tree Test Accuracy:  {dt_test_accuracy:.4f}")

Decision Tree Valid Log Loss: 2.0513
Decision Tree Valid Accuracy: 0.2636
--------------------------------------------------
Decision Tree Test Log Loss:  2.0450
Decision Tree Test Accuracy:  0.2671


### RANDOM FOREST

In [ ]:
%%time

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_valid_log_loss, rf_valid_accuracy, rf_test_log_loss, rf_test_accuracy = fit_and_evaluate(rf, flatten=True)

CPU times: total: 47.7 s
Wall time: 47.8 s


In [14]:
print(f"Random Forest Valid Log Loss: {rf_valid_log_loss:.4f}")
print(f"Random Forest Valid Accuracy: {rf_valid_accuracy:.4f}")
print("-" * 50)
print(f"Random Forest Test Log Loss:  {rf_test_log_loss:.4f}")
print(f"Random Forest Test Accuracy:  {rf_test_accuracy:.4f}")

Random Forest Valid Log Loss: 1.9571
Random Forest Valid Accuracy: 0.3302
--------------------------------------------------
Random Forest Test Log Loss:  1.9478
Random Forest Test Accuracy:  0.3397


### NEURAL NETWORK

In [5]:
class MLP(NNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
    ) -> None:
        input_size = X.shape[1]
        unique_classes = np.unique(y)
        output_size = len(unique_classes)

        self._get_network(input_size, output_size)

        loader = self._prepare_loader(X, y)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, 
            max_lr=self.learning_rate, 
            steps_per_epoch=len(loader), 
            epochs=self.epochs
        )

        self.train()
        for _ in range(self.epochs):
            for batch_X, batch_y in loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                optimizer.zero_grad()
                
                preds = self(batch_X)
                loss = criterion(preds, batch_y.to(torch.long).to(self.device).squeeze())
                
                loss.backward()
                optimizer.step()
                scheduler.step()
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        X_t = torch.from_numpy(X).to(torch.float32).to(self.device)
        self.eval()
        with torch.no_grad():
            logits = self.forward(X_t)
            predictions = torch.softmax(logits, dim=1)
        
        return predictions.cpu().numpy()

In [6]:
%%time

mlp = MLP(epochs=5, learning_rate=0.01, hidden_size=1024, batch_size=1024)
mlp_valid_log_loss, mlp_valid_accuracy, mlp_test_log_loss, mlp_test_accuracy = fit_and_evaluate(mlp, flatten=True)

CPU times: total: 11.9 s
Wall time: 5.73 s


In [7]:
print(f"Neural Network Valid Log Loss: {mlp_valid_log_loss:.4f}")
print(f"Neural Network Valid Accuracy: {mlp_valid_accuracy:.4f}")
print("-" * 50)
print(f"Neural Network Test Log Loss:  {mlp_test_log_loss:.4f}")
print(f"Neural Network Test Accuracy:  {mlp_test_accuracy:.4f}")

Neural Network Valid Log Loss: 1.7113
Neural Network Valid Accuracy: 0.3829
--------------------------------------------------
Neural Network Test Log Loss:  1.7035
Neural Network Test Accuracy:  0.3948


### XGBOOST

In [ ]:
%%time

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.25,
    early_stopping_rounds=10,
    max_depth=5,
    random_state=42
)

xt, xv, xte = X_train, X_valid, X_test

xt = xt.reshape(xt.shape[0], -1)
xv = xv.reshape(xv.shape[0], -1)
xte = xte.reshape(xte.shape[0], -1)

xgb.fit(xt, y_train, eval_set=[(xv, y_valid)])

xgb_valid_prob_preds = xgb.predict_proba(xv)
xgb_valid_class_preds = np.argmax(valid_prob_preds, axis=1)

xgb_valid_log_loss = log_loss(y_valid, xgb_valid_prob_preds, labels=np.unique(y_train))
xgb_valid_accuracy = accuracy_score(y_valid, xgb_valid_class_preds)

xgb_test_prob_preds = xgb.predict_proba(xte)
xgb_test_class_preds = np.argmax(xgb_test_prob_preds, axis=1)

xgb_test_log_loss = log_loss(y_test, xgb_test_prob_preds, labels=np.unique(y_train))
xgb_test_accuracy = accuracy_score(y_test, xgb_test_class_preds)

[0]	validation_0-mlogloss:2.08648
[1]	validation_0-mlogloss:1.97252
[2]	validation_0-mlogloss:1.89647
[3]	validation_0-mlogloss:1.83971
[4]	validation_0-mlogloss:1.79524
[5]	validation_0-mlogloss:1.75873
[6]	validation_0-mlogloss:1.72646
[7]	validation_0-mlogloss:1.70062
[8]	validation_0-mlogloss:1.67949
[9]	validation_0-mlogloss:1.66376
[10]	validation_0-mlogloss:1.64776
[11]	validation_0-mlogloss:1.63261
[12]	validation_0-mlogloss:1.62076
[13]	validation_0-mlogloss:1.60936
[14]	validation_0-mlogloss:1.59956
[15]	validation_0-mlogloss:1.58842
[16]	validation_0-mlogloss:1.57646
[17]	validation_0-mlogloss:1.56712
[18]	validation_0-mlogloss:1.56023
[19]	validation_0-mlogloss:1.55322
[20]	validation_0-mlogloss:1.54641
[21]	validation_0-mlogloss:1.54086
[22]	validation_0-mlogloss:1.53594
[23]	validation_0-mlogloss:1.52951
[24]	validation_0-mlogloss:1.52360
[25]	validation_0-mlogloss:1.51899
[26]	validation_0-mlogloss:1.51290
[27]	validation_0-mlogloss:1.50997
[28]	validation_0-mlogloss:1.5

In [20]:
print(f"XGBoost Valid Log Loss: {valid_log_loss:.4f}")
print(f"XGBoost Valid Accuracy: {valid_accuracy:.4f}")
print("-" * 50)
print(f"XGBoost Test Log Loss:  {test_log_loss:.4f}")
print(f"XGBoost Test Accuracy:  {test_accuracy:.4f}")

XGBoost Valid Log Loss: 1.4155
XGBoost Valid Accuracy: 0.5152
--------------------------------------------------
XGBoost Test Log Loss:  1.3960
XGBoost Test Accuracy:  0.5177


### BIG CNN

Total parameters: 259,914

Small CNN total parameters: 16,986

In [4]:
class CNN(CNNRegressor):
    def __init__(self, **hyperparameters):
        super().__init__(**hyperparameters)

    def fit(self, X, y):
        
        in_channels = X.shape[1]
        output_size = int(np.max(y)) + 1

        image_size = X.shape[-1]

        conv1_out = image_size - self.kernel_size + 1
        pool1_out = conv1_out // self.pool_size

        conv2_out = pool1_out - self.kernel_size + 1
        pool2_out = conv2_out // self.pool_size

        linear_input = self.channels[1] * pool2_out * pool2_out
        
        self._get_network(in_channels, linear_input, output_size)

        loader = self._prepare_loader(X, y)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), self.learning_rate)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, 
            max_lr=self.learning_rate, 
            steps_per_epoch=len(loader), 
            epochs=self.epochs
        )

        self.train()
        for epoch in range(self.epochs):
            running_loss = 0.0
            for batch_X, batch_y in loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()
                scheduler.step()

                running_loss += loss.item() * batch_X.size(0)
            
            epoch_loss = running_loss / len(loader.dataset)
            print(f"Epoch [{epoch+1}/{self.epochs}] - Training Loss: {epoch_loss:.4f}")
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        X_t = torch.from_numpy(X).to(torch.float32).to(self.device)
        self.eval()
        with torch.no_grad():
            logits = self.forward(X_t)
            predictions = torch.softmax(logits, dim=1)
        
        return predictions.cpu().numpy()

In [9]:
%%time

big_cnn = CNN(epochs=5, learning_rate=0.01, channels=[32, 64], kernel_size=5, pool_size=2, hidden_size=128, batch_size=1024)
big_cnn_valid_log_loss, big_cnn_valid_accuracy, big_cnn_test_log_loss, big_cnn_test_accuracy = fit_and_evaluate(big_cnn, flatten=False)

CPU times: total: 0 ns
Wall time: 9.06 μs


NameError: name 'CNN' is not defined

In [10]:
print(f"Big CNN Valid Log Loss: {big_cnn_valid_log_loss:.4f}")
print(f"Big CNN Valid Accuracy: {big_cnn_valid_accuracy:.4f}")
print("-" * 50)
print(f"Big CNN Test Log Loss:  {big_cnn_test_log_loss:.4f}")
print(f"Big CNN Test Accuracy:  {big_cnn_test_accuracy:.4f}")

Big CNN Valid Log Loss: 0.8075
Big CNN Valid Accuracy: 0.7185
--------------------------------------------------
Big CNN Test Log Loss:  0.8128
Big CNN Test Accuracy:  0.7213


### GRADIENT BOOSTING (BIG CNN)

In [4]:
%%time

nn_gb = GBClassifier(
    n_estimators=25,
    learning_rate=0.5,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=0.5,
    early_stopping_rounds=10,
    weak_learner_key="convolutional_neural_network",
    weak_learner_config={
        "epochs": 10,
        "learning_rate": 0.001,
        "channels": [32, 64],
        "kernel_size": 5,
        "pool_size": 2,
        "hidden_size": 128,
        "batch_size": 256
    }
)

nn_gb.fit(X_train, y_train, X_valid, y_valid)

nn_gb_valid_prob_preds = nn_gb.predict_proba(X_valid)
nn_gb_valid_class_preds = np.argmax(nn_gb_valid_prob_preds, axis=1)

nn_gb_valid_log_loss = log_loss(y_valid, nn_gb_valid_prob_preds, labels=np.unique(y_train))
nn_gb_valid_accuracy = accuracy_score(y_valid, nn_gb_valid_class_preds)

nn_gb_test_prob_preds = nn_gb.predict_proba(X_test)
nn_gb_test_class_preds = np.argmax(nn_gb_test_prob_preds, axis=1)

nn_gb_test_log_loss = log_loss(y_test, nn_gb_test_prob_preds, labels=np.unique(y_train))
nn_gb_test_accuracy = accuracy_score(y_test, nn_gb_test_class_preds)

Epoch [1/10] - Training Loss: 0.0808
Epoch [2/10] - Training Loss: 0.0014
Epoch [3/10] - Training Loss: 0.0006
Epoch [4/10] - Training Loss: 0.0007
Epoch [5/10] - Training Loss: 0.0007
Epoch [6/10] - Training Loss: 0.0005
Epoch [7/10] - Training Loss: 0.0005
Epoch [8/10] - Training Loss: 0.0005
Epoch [9/10] - Training Loss: 0.0005
Epoch [10/10] - Training Loss: 0.0004


2026-04-27 19:44:15,208 - INFO - Iteration: 1 | Validation Log Loss: 2.302850 | Validation Accuracy: 0.093300


CPU times: total: 1min 14s
Wall time: 1min 7s


KeyboardInterrupt: 

In [16]:
print(f"NN GB Valid Log Loss: {nn_gb_valid_log_loss:.4f}")
print(f"NN GB Valid Accuracy: {nn_gb_valid_accuracy:.4f}")
print("-" * 50)
print(f"NN GB Test Log Loss:  {nn_gb_test_log_loss:.4f}")
print(f"NN GB Test Accuracy:  {nn_gb_test_accuracy:.4f}")

NN GB Valid Log Loss: 2.3021
NN GB Valid Accuracy: 0.1140
--------------------------------------------------
NN GB Test Log Loss:  2.3020
NN GB Test Accuracy:  0.1160


### SMALL CNN

In [13]:
small_cnn = CNN(epochs=100, learning_rate=0.001, channels=[8, 16], kernel_size=5, pool_size=2, hidden_size=32, batch_size=256)
small_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

raw_preds = small_cnn.predict(test) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch: 1 | Validation Log Loss: 1.7002 | Validation Accuracy: 0.3900
Epoch: 2 | Validation Log Loss: 1.5622 | Validation Accuracy: 0.4384
Epoch: 3 | Validation Log Loss: 1.5171 | Validation Accuracy: 0.4510
Epoch: 4 | Validation Log Loss: 1.4619 | Validation Accuracy: 0.4844
Epoch: 5 | Validation Log Loss: 1.4579 | Validation Accuracy: 0.4823
Epoch: 6 | Validation Log Loss: 1.4371 | Validation Accuracy: 0.4888
Epoch: 7 | Validation Log Loss: 1.4012 | Validation Accuracy: 0.5038
Epoch: 8 | Validation Log Loss: 1.3744 | Validation Accuracy: 0.5152
Epoch: 9 | Validation Log Loss: 1.3396 | Validation Accuracy: 0.5242
Epoch: 10 | Validation Log Loss: 1.3606 | Validation Accuracy: 0.5152
Epoch: 11 | Validation Log Loss: 1.3355 | Validation Accuracy: 0.5284
Epoch: 12 | Validation Log Loss: 1.3025 | Validation Accuracy: 0.5427
Epoch: 13 | Validation Log Loss: 1.2748 | Validation Accuracy: 0.5514
Epoch: 14 | Validation Log Loss: 1.2942 | Validation Accuracy: 0.5471
Epoch: 15 | Validation Log Lo